# UC1_AI_Data_Quality_Reporter

```
=============================================================================
UC1 — AI DATA QUALITY REPORTER
=============================================================================
Reads from : globalmart.silver.quarantine_customers
             globalmart.silver.quarantine_orders
             globalmart.silver.quarantine_transactions
Writes to  : globalmart.gold.dq_audit_report
             globalmart.gold.pipeline_run_log
Model      : databricks-gpt-oss-20b via Databricks AI Gateway
=============================================================================
```

In [0]:
# %pip install openai
# dbutils.library.restartPython()

In [0]:
import time
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType
)
from openai import OpenAI

# AI Gateway connection — same pattern as reference notebooks
client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url="https://4092199907892374.ai-gateway.cloud.databricks.com/mlflow/v1/"
)

MODEL_NAME   = "databricks-gpt-oss-20b"
NOTEBOOK_NAME = "UC1_AI_Data_Quality_Reporter"

print("LLM client ready.")
print(f"Model : {MODEL_NAME}")
print(f"Run started at : {datetime.now().isoformat()}")

In [0]:
def extract_text(response) -> str:
    """
    databricks-gpt-oss-20b returns a structured list:
      [{"type": "reasoning", "summary": [...]}, {"type": "text", "text": "..."}]
    We extract only the 'text' block.
    If the response is already a plain string, return it directly.
    """
    content = response.choices[0].message.content
    if isinstance(content, list):
        parts = [
            block["text"]
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        ]
        return " ".join(parts).strip()
    return content.strip()


def call_llm(prompt: str, system_msg: str, retries: int = 3, wait: int = 2) -> str:
    """
    Calls the LLM with exponential backoff retry.
    Returns LLM_UNAVAILABLE:<error> string on total failure
    so the pipeline continues rather than crashing.
    """
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=2000,
                temperature=0.3
            )
            return extract_text(response)
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(wait * (attempt + 1))
            else:
                return f"LLM_UNAVAILABLE: {str(e)}"


def validate_llm_output(text: str, min_words: int = 20) -> dict:
    """
    Validates LLM output before writing to Delta.
    Catches truncation failures and refusal responses.
    Returns a dict with the text, quality_flag (PASS/REVIEW), and issues list.
    """
    refusal_phrases = ["i cannot", "as an ai", "i don't have", "i'm unable", "i am unable"]
    issues = []

    if len(text.split()) < min_words:
        issues.append("too_short")
    if text.startswith("LLM_UNAVAILABLE"):
        issues.append("llm_call_failed")
    if any(phrase in text.lower() for phrase in refusal_phrases):
        issues.append("refusal_detected")

    return {
        "text":         text,
        "quality_flag": "PASS" if not issues else "REVIEW",
        "issues":       ", ".join(issues) if issues else "none"
    }


print("Helper functions defined.")

In [0]:
# Each quarantine table must contain at minimum:
#   issue_type      — category of the quality problem e.g. "missing_value"
#   field_affected  — column name that failed the rule e.g. "customer_id"
# Adjust column names here if your Silver layer uses different naming.

df_q_customers    = spark.table("globalmart.silver.customers_quarantine")
df_q_orders       = spark.table("globalmart.silver.orders_quarantine")
df_q_transactions = spark.table("globalmart.silver.transactions_quarantine")
df_q_products    = spark.table("globalmart.silver.products_quarantine")
df_q_returns      = spark.table("globalmart.silver.returns_quarantine")
df_q_vendors = spark.table("globalmart.silver.vendors_quarantine")

print("Quarantine table row counts:")
print(f"  customers    : {df_q_customers.count()}")
print(f"  orders       : {df_q_orders.count()}")
print(f"  transactions : {df_q_transactions.count()}")
print(f"  customers    : {df_q_products.count()}")
print(f"  orders       : {df_q_returns.count()}")
print(f"  transactions : {df_q_vendors.count()}")

In [0]:
# Check the last run timestamp from the audit report.
# On first run this returns None and we process everything.
# On subsequent runs, only new quarantine records since last run are processed.

try:
    last_run_ts = spark.sql("""
        SELECT MAX(generated_at) as last_ts
        FROM globalmart.gold.dq_audit_report
    """).collect()[0]["last_ts"]
    print(f"Last successful run at: {last_run_ts}. Processing new records only.")
except Exception:
    last_run_ts = None
    print("No previous run found. Processing all quarantine records.")

def apply_watermark(df, ts_col="_quarantine_timestamp"):
    """Filter to records newer than last run. If no watermark, return full df."""
    if last_run_ts is not None and ts_col in df.columns:
        return df.filter(F.col(ts_col) > F.lit(last_run_ts))
    return df

df_q_customers    = apply_watermark(df_q_customers)
df_q_orders       = apply_watermark(df_q_orders)
df_q_transactions = apply_watermark(df_q_transactions)

print("Post-watermark row counts:")
print(f"  customers    : {df_q_customers.count()}")
print(f"  orders       : {df_q_orders.count()}")
print(f"  transactions : {df_q_transactions.count()}")

In [0]:
# NEVER pass individual rows to the LLM.
# Group first — one explanation per issue group, not per row.
# This keeps LLM calls minimal and output quality high.

def aggregate_quarantine(df, entity_name: str):
    return (
        df.groupBy("_issue_type", "_expectation_name")
          .agg(F.count("*").alias("rejected_count"))
          .withColumn("entity", F.lit(entity_name))
          .withColumnRenamed("_expectation_name", "field_affected")
          .withColumnRenamed("_issue_type",       "issue_type")
    )

dq_summary = (
    aggregate_quarantine(df_q_customers,    "customers")
    .union(aggregate_quarantine(df_q_orders,       "orders"))
    .union(aggregate_quarantine(df_q_transactions, "transactions"))
    .collect()
)

print(f"\nTotal issue groups to explain: {len(dq_summary)}")
for row in dq_summary:
    print(f"  [{row['entity']}] field='{row['field_affected']}'"
          f" | issue='{row['issue_type']}' | count={row['rejected_count']}")

In [0]:
# Classifies each issue group as CRITICAL, HIGH, or MEDIUM
# so the finance team can prioritize the audit report.

def classify_severity(entity: str, issue_type: str, count: int) -> str:
    """
    CRITICAL : Missing primary key on any entity, or any issue with count > 100
    HIGH     : Invalid reference, negative amounts, or count > 30
    MEDIUM   : Format issues, invalid ranges, or everything else
    """
    critical_issues = {"missing_value", "null_primary_key", "missing_customer_id", "missing_order_id"}
    high_issues     = {"invalid_reference", "negative_amount", "orphaned_record", "duplicate_primary_key"}

    if issue_type.lower() in critical_issues or count > 100:
        return "CRITICAL"
    elif issue_type.lower() in high_issues or count > 30:
        return "HIGH"
    else:
        return "MEDIUM"

print("Severity classifier ready.")

In [0]:

def build_dq_prompt(entity: str, field: str, issue_type: str, count: int, severity: str) -> str:
    return f"""You are preparing a data quality audit report for globalmart's external finance audit.

globalmart is a national retail chain that consolidated data from 6 disconnected regional systems.
During Silver layer harmonization, records were rejected and quarantined when they failed quality rules.

The following issue group has been identified:
  Entity           : {entity}
  Field affected   : {field}
  Issue type       : {issue_type}
  Rejected records : {count}
  Severity         : {severity}

Write 3-4 sentences in plain English that answer:
1. What the problem is and what value or pattern triggered the rejection
2. Why this record type cannot be accepted into the analytics layer
3. What business report, audit figure, or operational decision is at risk

No bullet points. No markdown. No headers. Write as connected prose for a finance director
who does not know Python or SQL."""


SYSTEM_MSG_UC1 = (
    "You are a data quality analyst preparing plain-English audit summaries "
    "for a non-technical finance director at globalmart retail. "
    "Write in plain professional English. No jargon, no markdown, no bullet points."
)

print("Prompt builder ready.")

In [0]:

audit_records  = []
llm_call_count = 0
review_count   = 0

for row in dq_summary:
    entity    = row["entity"]
    field     = row["field_affected"]
    issue     = row["issue_type"]
    count     = row["rejected_count"]
    severity  = classify_severity(entity, issue, count)

    prompt      = build_dq_prompt(entity, field, issue, count, severity)
    raw_output  = call_llm(prompt, SYSTEM_MSG_UC1)
    validated   = validate_llm_output(raw_output)
    llm_call_count += 1

    if validated["quality_flag"] == "REVIEW":
        review_count += 1

    audit_records.append({
        "entity":           entity,
        "field_affected":   field,
        "issue_type":       issue,
        "rejected_count":   int(count),
        "severity":         severity,
        "ai_explanation":   validated["text"],
        "quality_flag":     validated["quality_flag"],
        "quality_issues":   validated["issues"],
        "generated_at":     datetime.now().isoformat()
    })

    print("=" * 70)
    print(f"[{severity}] {entity} | {field} | {issue} | {count} records")
    print(f"Quality flag : {validated['quality_flag']}")
    print(f"\nEXPLANATION :\n{validated['text']}")
    print("=" * 70 + "\n")

print(f"Generated {len(audit_records)} explanations.")
print(f"LLM calls made  : {llm_call_count}")
print(f"Outputs flagged for review : {review_count}")

In [0]:

schema = StructType([
    StructField("entity",           StringType(),  True),
    StructField("field_affected",   StringType(),  True),
    StructField("issue_type",       StringType(),  True),
    StructField("rejected_count",   IntegerType(), True),
    StructField("severity",         StringType(),  True),
    StructField("ai_explanation",   StringType(),  True),
    StructField("quality_flag",     StringType(),  True),
    StructField("quality_issues",   StringType(),  True),
    StructField("generated_at",     StringType(),  True)
])

df_audit = spark.createDataFrame(audit_records, schema=schema)

(
    df_audit.write
    .format("delta")
    .mode("append")                         # append — preserve audit history
    .option("mergeSchema", "true")
    .saveAsTable("globalmart.gold.dq_audit_report")
)

print("Audit report written to globalmart.gold.dq_audit_report")
display(
    spark.table("globalmart.gold.dq_audit_report")
    .orderBy(
        F.when(F.col("severity") == "CRITICAL", 1)
         .when(F.col("severity") == "HIGH", 2)
         .otherwise(3)
    )
)

In [0]:

run_log = [{
    "notebook":          NOTEBOOK_NAME,
    "run_timestamp":     datetime.now().isoformat(),
    "records_processed": len(dq_summary),
    "llm_calls_made":    llm_call_count,
    "outputs_flagged":   review_count,
    "status":            "SUCCESS" if review_count < llm_call_count else "PARTIAL_REVIEW",
    "notes":             f"watermark_applied={last_run_ts is not None}"
}]

(
    spark.createDataFrame(run_log)
    .write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("globalmart.gold.pipeline_run_log")
)

print(f"\nRun log written to globalmart.gold.pipeline_run_log")
print(f"Status  : {run_log[0]['status']}")
print(f"Outputs requiring human review : {review_count}")